In [ ]:
# Data processing
# ==============================================================================
import numpy as np
import pandas as pd
import polars
from skforecast.datasets import fetch_dataset
import sys

# Plots
# ==============================================================================
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.graphics.tsaplots import plot_pacf
import plotly.graph_objects as go
import plotly.io as pio
import plotly.offline as poff
pio.templates.default = "seaborn"
poff.init_notebook_mode(connected=True)

pd.set_option('display.float_format', '{:.6f}'.format)

In [ ]:
df_pred = pd.read_parquet('C:/Users/J.Heuvelmans/OneDrive - Brain Research Center/Documenten/EAISI/2024Applications/Group4B/Data/processed/df_pred20241120.parquet', engine='pyarrow')
# df_test = pd.read_parquet('C:/Users/J.Heuvelmans/OneDrive - Brain Research Center/Documenten/EAISI/2024Supermarket/Data/processed/df_test_with_forecasts_20241120.parquet', engine='pyarrow')
# df_val = pd.read_parquet('C:/Users/J.Heuvelmans/OneDrive - Brain Research Center/Documenten/EAISI/2024Supermarket/Data/processed/df_val_with_forecasts_20241120.parquet', engine='pyarrow')

In [ ]:
df_pred.head()

In [ ]:
df = df_pred
df.head()

In [ ]:
# df = df[df['week_number_cum'] != 242]

In [ ]:
def calculate_bias_acc(df, prediction, actual):
    """
    Calculate bias, accuracy, and adjusted bias for predictions.
    
    Parameters:
        df (pd.DataFrame): The DataFrame containing the prediction and actual data.
        prediction (str): Column name for the predictions.
        actual (str): Column name for the actual values.

    Returns:
        pd.DataFrame: The DataFrame with new columns for bias, accuracy, and adjusted bias.
    """
    # Calculate bias
    df[f'bias_{prediction}'] = df[prediction] - df[actual]
    
    # Calculate accuracy
    df[f'acc_{prediction}'] = np.where(
        (df[actual] == 0) & (df[prediction] != 0),
        np.nan,
        (1 - np.abs(df[f'bias_{prediction}']) / df[actual]) * 100
    )
    
    # Calculate adjusted bias
    df[f'adj_bias_{prediction}'] = (df[f'bias_{prediction}'] * 2 / 3).where(
        df[f'bias_{prediction}'] >= 0, 
        df[f'bias_{prediction}'] * -1 / 3
    )
    
    return df


In [ ]:
list_pred = ['y_xgb', 'y_mean', 'y_naive']

# Apply the function for every prediction column in list_pred
for pred in list_pred:
    df = calculate_bias_acc(df, pred, 'unit_sales')

# Display the updated DataFrame
df

In [ ]:
def calculate_metrics(df):
    # Calculate mean accuracies
    accuracy_xgb = np.mean(df['acc_y_xgb'])
    accuracy_naive = np.mean(df['acc_y_naive'])
    accuracy_mean = np.mean(df['acc_y_mean'])
        
    # Calculate bias and positive/negative bias for XGB and Naive models
    bias_xgb = np.mean(df['bias_y_xgb'])
    positive_bias_xgb = np.mean(df['bias_y_xgb'][df['bias_y_xgb'] > 0])
    negative_bias_xgb = np.mean(df['bias_y_xgb'][df['bias_y_xgb'] < 0])
    
    bias_naive = np.mean(df['bias_y_naive'])
    positive_bias_naive = np.mean(df['bias_y_naive'][df['bias_y_naive'] > 0])
    negative_bias_naive = np.mean(df['bias_y_naive'][df['bias_y_naive'] < 0])

    bias_mean = np.mean(df['bias_y_mean'])
    positive_bias_mean = np.mean(df['bias_y_mean'][df['bias_y_mean'] > 0])
    negative_bias_mean = np.mean(df['bias_y_mean'][df['bias_y_mean'] < 0])

    # Calculate sum of adjusted bias
    adj_bias_xgb = np.sum(df['adj_bias_y_xgb'])
    adj_bias_naive = np.sum(df['adj_bias_y_naive'])
    adj_bias_mean = np.sum(df['adj_bias_y_mean'])
    diff_xgb_naive = adj_bias_naive - adj_bias_xgb
    diff_xgb_mean = adj_bias_naive - adj_bias_mean
    
    
    # Store results in a dictionary
    results = {
        'Metric': [
            'Mean Accuracy XGB', 'Mean Accuracy Naive',
            'Mean Accuracy Mean',
            'Bias XGB', 'Positive Bias XGB', 'Negative Bias XGB', 
            'Bias Naive', 'Positive Bias Naive', 'Negative Bias Naive',
            'Bias Mean', 'Positive Bias Mean', 'Negative Bias Mean',
            'Money XGB', 'Money Naive', 'Money Mean',
            'Difference Naive - XGB', 'Difference Naive - Mean'
        ],
        'Value': [
            accuracy_xgb, accuracy_naive, 
            accuracy_mean,
            bias_xgb, positive_bias_xgb, negative_bias_xgb,
            bias_naive, positive_bias_naive, negative_bias_naive,
            bias_mean, positive_bias_mean, negative_bias_mean,
            adj_bias_xgb, adj_bias_naive, adj_bias_mean,
            diff_xgb_naive, diff_xgb_mean
        ]
    }
    
    # Convert results dictionary to a DataFrame
    results_df = pd.DataFrame(results)
    
    return results_df

In [ ]:
def calculate_metrics_grouped(df, column):
    store_stats = []
    
    # Iterate over each group's data
    for store, group in df.groupby(column):
        # # Skip calculation if the group value is 0 or missing (NaN)
        # if store == 0 or pd.isna(store):
        #     continue
        
        # Calculate statistics for the current group using the existing function
        df_metrics = calculate_metrics(group)
        
        # Add the column value (e.g., store number) to each metric's result for reference
        df_metrics[column] = store
        
        # Append each group's result to the list
        store_stats.append(df_metrics)
    
    # Concatenate all group statistics into a single DataFrame
    store_means = pd.concat(store_stats).reset_index(drop=True)

    # Pivot the DataFrame to make column values into columns and Metric rows
    store_metrics_pivot = store_means.pivot(index='Metric', columns=column, values='Value')

    # Drop columns with all NA values
    store_metrics_pivot = store_metrics_pivot.dropna(axis=1, how='all')

    return store_metrics_pivot

In [ ]:
def summarize_store_data(df, group_column, summary_columns, operation='sum'):
    if operation == 'sum':
        grouped_df = df.groupby(group_column)[summary_columns].sum().reset_index()
    elif operation == 'mean':
        grouped_df = df.groupby(group_column)[summary_columns].mean().reset_index()
    else:
        raise ValueError("Invalid operation. Choose 'sum' or 'mean'.")
    
    return grouped_df

In [ ]:
def sorted_histogram(dict_df, group_names):
    """
    Plots sorted histograms for the first row (sorted) and unsorted histograms for others.
    The bars for each row are plotted next to each other.

    Parameters:
        dict_df (dict): A dictionary where keys are group names and values are DataFrames.
        group_names (list): A list of row names to plot.

    Returns:
        None
    """
    # Iterate over each DataFrame in the dictionary
    for key, df in dict_df.items():
        print(f"Processing group: {key}")
        
        # Create a new figure for each group
        plt.figure(figsize=(12, 8))
        
        # Get the number of elements in the first row to define x positions
        # (Assumes all rows have the same length, but you could adjust for mismatched lengths)
        num_elements = len(df.loc[group_names[0]])
        
        # Initialize the width of each bar and set the starting position for the bars
        bar_width = 0.2
        index = np.arange(num_elements)  # positions for the x-axis
        
        # Loop through each group name in the provided list
        for i, group_name in enumerate(group_names):
            # Ensure the row exists in the DataFrame
            if group_name not in df.index:
                raise ValueError(f"Row '{group_name}' not found in the DataFrame index of group '{key}'.")
            
            # Extract the specified row and convert to a Series
            difference_row = df.loc[group_name]
            
            # Sort the first group (group_names[0]) only
            if i == 0:
                difference_row = difference_row.sort_values(ascending=False)
            
            # Plot the histogram for each group_name, shifting bars by `i * bar_width`
            # This ensures the bars for different rows are placed next to each other
            plt.bar(index + i * bar_width, difference_row, width=bar_width, label=group_name)
        
        # Add title, labels, and grid
        plt.title(f'Saved money compared to naive, per: {key}', fontsize=14)
        plt.xlabel({key}, fontsize=12)
        plt.ylabel('Saved money', fontsize=12)
        
        # Adjust x-ticks to correspond to the center of the grouped bars
        plt.xticks(index + bar_width * (len(group_names) - 1) / 2, difference_row.index.astype(str), rotation=90)
        
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.legend(title="Rows", fontsize=10)
        
        # Adjust layout and show the plot
        plt.tight_layout()
        plt.show()

In [ ]:
def plot_sales_comparison(df, filter_column, filter_value):
    """
    Plots a comparison of predicted sales (y_xgb), actual sales (unit_sales), and naive predictions (y_naive)
    over weeks for a specified store (or any other column-based filter).

    Parameters:
    df_pred (pd.DataFrame): DataFrame containing the data.
    filter_column (str): The column name to filter by (e.g., 'store_nbr').
    filter_value (int or str): The value to filter the specified column by (e.g., a store number).
    """
    # Step 1: Filter data for the specified column and value
    grouped_df = df[df[filter_column] == filter_value]

    # Step 2: Group by 'week_number_cum' and compute the sum for each of the columns
    grouped_df = grouped_df.groupby('week_number_cum')[['y_xgb', 'unit_sales', 'y_naive', 'y_mean']].sum().reset_index()

    # Step 3: Plot the results
    plt.figure(figsize=(12, 6))
    plt.plot(grouped_df['week_number_cum'], grouped_df['y_xgb'], marker='o', label='XGB prediction (y_xgb)', color='blue')
    plt.plot(grouped_df['week_number_cum'], grouped_df['unit_sales'], marker='o', label='Actual Sales (unit_sales)', color='black')
    plt.plot(grouped_df['week_number_cum'], grouped_df['y_naive'], marker='o', label='Naive prediction (y_naive)', color='red')
    plt.plot(grouped_df['week_number_cum'], grouped_df['y_mean'], marker='o', label='Average mean prediction (y_mean)', color='green')

    # Add titles and labels
    plt.title(f'Sum of Predicted sales vs Actual sales Over Weeks for {filter_column} = {filter_value}')
    plt.xlabel('Week Number Cumulative')
    plt.ylabel('Sales')
    plt.xticks(grouped_df['week_number_cum'])  # Set x-ticks to show all weeks
    plt.legend()
    plt.grid()

    # Show the plot
    plt.tight_layout()
    plt.show()


In [ ]:
    grouped_df = df
    # Step 2: Group by 'week_number_cum' and compute the sum for each of the columns
    grouped_df = grouped_df.groupby('week_number_cum')[['y_xgb', 'unit_sales', 'y_naive', 'y_mean']].sum().reset_index()

    # Step 3: Plot the results
    plt.figure(figsize=(12, 12))
    plt.plot(grouped_df['week_number_cum'], grouped_df['y_xgb'], marker='o', label='XGB prediction (y_xgb)', color='blue')
    plt.plot(grouped_df['week_number_cum'], grouped_df['unit_sales'], marker='o', label='Actual Sales (unit_sales)', color='black')
    plt.plot(grouped_df['week_number_cum'], grouped_df['y_naive'], marker='o', label='Naive prediction (y_naive)', color='red')
    plt.plot(grouped_df['week_number_cum'], grouped_df['y_mean'], marker='o', label='Average mean prediction (y_mean)', color='green')

    # Add titles and labels
    plt.title(f'Sum of Predicted sales vs Actual sales Over Weeks')
    plt.xlabel('Week Number Cumulative')
    plt.ylabel('Sales')
    plt.xticks(grouped_df['week_number_cum'])  # Set x-ticks to show all weeks
    plt.legend()
    plt.grid()

    # Show the plot
    plt.tight_layout()
    plt.show()

In [ ]:
# Choose from list_grouping a column, get unique values, and create line graphs with total actual sales, predicted sales and naive prediction

column = 'item_family'
# Get unique values in the current column
unique_values = df[column].unique()
    
# Loop through each unique value in the column
for value in unique_values:
    # Call the plot_sales_comparison function
    plot_sales_comparison(df, filter_column=column, filter_value=value)


In [ ]:
df_metrics = calculate_metrics(df)
df_metrics

In [ ]:
list_grouping = ['store_type', 'store_cluster', 'store_nbr', 'perishable', 'item_family', 'item_class', 'week_number_cum']

# Initialize an empty dictionary
dict_df_metrics = {}

for i in list_grouping:
    # Calculate metrics for the current grouping
    df_metrics_grouped = calculate_metrics_grouped(df, i)
    dict_df_metrics[i] = df_metrics_grouped  # Add the resulting DataFrame to the dictionary with `i` as the key
    print(df_metrics_grouped)

In [ ]:
for i in list_grouping:
    df_sum = summarize_store_data(df, group_column=i, 
                              summary_columns=['unit_sales', 'y_naive', 'y_xgb', 'y_mean', 
                                               'acc_y_xgb', 'acc_y_naive', 'acc_y_mean', 
                                               'adj_bias_y_xgb', 'adj_bias_y_naive', 'adj_bias_y_mean'],
                              operation='sum')
    print(df_sum)

In [ ]:
for i in list_grouping:
    df_mean = summarize_store_data(df, group_column=i, 
                              summary_columns=['unit_sales', 'y_naive', 'y_xgb', 'y_mean', 
                                               'acc_y_xgb', 'acc_y_naive', 'acc_y_mean', 
                                               'adj_bias_y_xgb', 'adj_bias_y_naive', 'adj_bias_y_mean'],
                              operation='mean')
    print(df_mean)

In [ ]:

# Example usage with a list of group names
group_names = ['Difference Naive - XGB', 'Difference Naive - Mean']
sorted_histogram(dict_df_metrics, group_names)

In [ ]:
df_week_number_cum = calculate_metrics_grouped(df, 'week_number_cum')

# Assuming df_plot is the DataFrame you are working with
df_plot = df_week_number_cum

# Extract the row for "Difference Naive - XGB" and transpose it to work with it as a Series
difference_row_xgb = df_plot.loc['Difference Naive - XGB'].transpose()

# Extract the row for "Difference Naive - Mean" and transpose it to work with it as a Series
difference_row_mean = df_plot.loc['Difference Naive - Mean'].transpose()

# Ensure the index represents week numbers and the values are numeric
week_numbers = df_plot.columns.astype(int)  # Assuming column names are week numbers

# Convert the values for both rows to numeric, coercing any errors to NaN
difference_values_xgb = pd.to_numeric(difference_row_xgb, errors='coerce')
difference_values_mean = pd.to_numeric(difference_row_mean, errors='coerce')

# Compute the cumulative sum of the difference values for both rows
cumulative_difference_xgb = difference_values_xgb.cumsum()
cumulative_difference_mean = difference_values_mean.cumsum()

# Plot the cumulative difference for both "Difference Naive - XGB" and "Difference Naive - Mean"
plt.figure(figsize=(12, 6))

# Plot the cumulative difference for "Difference Naive - XGB"
plt.plot(week_numbers, cumulative_difference_xgb, marker='o', label='Money saved by XGB model', color='blue')

# Plot the cumulative difference for "Difference Naive - Mean"
plt.plot(week_numbers, cumulative_difference_mean, marker='s', label='Money saved by moving average model', color='green')

# Add labels and title
plt.title('Cumulative Money Saved by model', fontsize=14)
plt.xlabel('Week Number', fontsize=12)
plt.ylabel('Total Money Saved', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

# Show the plot
plt.tight_layout()
plt.show()
